In [1]:
import pandas as pd
import numpy as np
import os
import warnings

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import ExtraTreesRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")


In [2]:
train = pd.read_csv("data/train_df_1.csv")
test = pd.read_csv("data/test_df_1.csv")

test_ids = test["ID"]

print(train.shape, test.shape)

(20004, 12) (3334, 11)


In [3]:
def create_features(df):
    df = df.copy()

    dt = pd.to_datetime(df["Data"] + " " + df["Time"], format="%d-%m-%Y %H:%M:%S")
    sunrise = pd.to_datetime(df["Data"] + " " + df["TimeSunRise"], format="%d-%m-%Y %H:%M:%S")
    sunset = pd.to_datetime(df["Data"] + " " + df["TimeSunSet"], format="%d-%m-%Y %H:%M:%S")

    df["hour"] = dt.dt.hour + dt.dt.minute / 60
    df["minute_of_day"] = dt.dt.hour * 60 + dt.dt.minute
    df["day"] = dt.dt.day
    df["month"] = dt.dt.month
    df["dayofyear"] = dt.dt.dayofyear
    df["weekday"] = dt.dt.weekday

    df["sunrise_min"] = sunrise.dt.hour * 60 + sunrise.dt.minute
    df["sunset_min"] = sunset.dt.hour * 60 + sunset.dt.minute

    df["day_length"] = df["sunset_min"] - df["sunrise_min"]
    df["time_from_sunrise"] = df["minute_of_day"] - df["sunrise_min"]
    df["time_to_sunset"] = df["sunset_min"] - df["minute_of_day"]

    df["solar_progress"] = df["time_from_sunrise"] / df["day_length"]
    df["solar_angle"] = np.sin(np.pi * df["solar_progress"])
    df["solar_angle"] = df["solar_angle"].clip(0, None)

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    df["day_sin"] = np.sin(2 * np.pi * df["dayofyear"] / 365)
    df["day_cos"] = np.cos(2 * np.pi * df["dayofyear"] / 365)

    df["wind_sin"] = np.sin(np.deg2rad(df["WindDirection(Degrees)"]))
    df["wind_cos"] = np.cos(np.deg2rad(df["WindDirection(Degrees)"]))

    df["temp_humidity"] = df["Temperature"] * df["Humidity"]
    df["temp_pressure"] = df["Temperature"] * df["Pressure"]
    df["solar_temp"] = df["solar_angle"] * df["Temperature"]
    df["solar_humidity"] = df["solar_angle"] * df["Humidity"]

    df = df.drop(["ID", "Data", "Time", "TimeSunRise", "TimeSunSet"], axis=1)

    return df


X = create_features(train.drop("Radiation", axis=1))
X_test = create_features(test)

y = train["Radiation"]

# missing values
X = X.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

X = X.fillna(X.median())
X_test = X_test.fillna(X.median())


In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    print(f"\nFold {fold + 1}")

    X_train, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    lgb = LGBMRegressor(
        n_estimators=1200,
        learning_rate=0.03,
        num_leaves=64,
        max_depth=-1,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.2,
        reg_lambda=1.0,
        random_state=42 + fold,
        verbose=-1
    )

    et = ExtraTreesRegressor(
        n_estimators=500,
        max_features=0.9,
        min_samples_leaf=1,
        random_state=42 + fold,
        n_jobs=-1
    )

    lgb.fit(X_train, y_train)
    et.fit(X_train, y_train)

    pred_lgb = lgb.predict(X_val)
    pred_et = et.predict(X_val)

    val_pred = 0.65 * pred_lgb + 0.35 * pred_et

    oof[val_idx] = val_pred

    test_pred = 0.65 * lgb.predict(X_test) + 0.35 * et.predict(X_test)
    test_preds += test_pred / kf.n_splits

    rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    print("Fold RMSE:", rmse)



Fold 1
Fold RMSE: 81.10260723096323

Fold 2
Fold RMSE: 77.85992356801938

Fold 3
Fold RMSE: 72.83198211385444

Fold 4
Fold RMSE: 77.73014075143773

Fold 5
Fold RMSE: 82.66303208268016


In [5]:
final_rmse = np.sqrt(mean_squared_error(y, oof))
print("\nFinal CV RMSE:", final_rmse)


Final CV RMSE: 78.51023192219465


In [6]:
submission = pd.DataFrame({
    "ID": test_ids,
    "TARGET": test_preds
})

submission.to_csv("submission.csv", index=False)

submission.head()

,ID,TARGET
0,1,699.946048
1,2,2.201637
2,3,0.173820
3,4,2.901733
4,5,3.494494
